In [0]:
product=spark.read.csv("/Volumes/course_enrollment/inventory_schema/filestorage/products.csv").toDF("product_id","product_name","category","reorder_level")
warehouse=spark.read.csv("/Volumes/course_enrollment/inventory_schema/filestorage/warehouse.csv").toDF("warehouse_id","city","product_id","current_stock")

In [0]:
display(product)
display(warehouse)

product_id,product_name,category,reorder_level
P001,Wireless Mouse,Electronics,20
P002,Ergonomic Chair,Office Furniture,5
P003,Mechanical Keyboard,Electronics,15
P004,Desk Lamp,Office Supplies,10
P005,USB-C Cable,Electronics,30
P006,External Hard Drive,Electronics,12
P007,Monitor Stand,Office Supplies,8
P008,Bluetooth Speaker,Electronics,25
P009,Fountain Pen,Office Supplies,50
P010,Notebook Journal,Office Supplies,40


warehouse_id,city,product_id,current_stock
WH-01,New York Hub,P001,12
WH-01,New York Hub,P002,8
WH-01,New York Hub,P003,25
WH-02,Los Angeles Logistics,P004,3
WH-02,Los Angeles Logistics,P005,50
WH-03,Chicago Depot,P006,7
WH-03,Chicago Depot,P007,15
WH-04,Houston Fulfillment,P008,30
WH-04,Houston Fulfillment,P009,12
WH-05,Phoenix Storage,P010,45


In [0]:
from pyspark.sql.functions import when
joined_df=product.join(warehouse,product.product_id==warehouse.product_id)
display(joined_df)
joined_df=joined_df.withColumn("stock_status",when(joined_df.current_stock>joined_df.reorder_level,"Satisfied").otherwise("out of stock"))
final_df=joined_df.select("warehouse_id","product_name","city","stock_status")
display(final_df)


product_id,product_name,category,reorder_level,warehouse_id,city,product_id,current_stock
P001,Wireless Mouse,Electronics,20,WH-10,San Jose Annex,P001,45
P002,Ergonomic Chair,Office Furniture,5,WH-01,New York Hub,P002,8
P003,Mechanical Keyboard,Electronics,15,WH-01,New York Hub,P003,25
P004,Desk Lamp,Office Supplies,10,WH-02,Los Angeles Logistics,P004,3
P005,USB-C Cable,Electronics,30,WH-02,Los Angeles Logistics,P005,50
P006,External Hard Drive,Electronics,12,WH-03,Chicago Depot,P006,7
P007,Monitor Stand,Office Supplies,8,WH-03,Chicago Depot,P007,15
P008,Bluetooth Speaker,Electronics,25,WH-04,Houston Fulfillment,P008,30
P009,Fountain Pen,Office Supplies,50,WH-04,Houston Fulfillment,P009,12
P010,Notebook Journal,Office Supplies,40,WH-05,Phoenix Storage,P010,45


warehouse_id,product_name,city,stock_status
WH-01,Wireless Mouse,New York Hub,out of stock
WH-01,Ergonomic Chair,New York Hub,Satisfied
WH-01,Mechanical Keyboard,New York Hub,Satisfied
WH-02,Desk Lamp,Los Angeles Logistics,Satisfied
WH-02,USB-C Cable,Los Angeles Logistics,Satisfied
WH-03,External Hard Drive,Chicago Depot,Satisfied
WH-03,Monitor Stand,Chicago Depot,out of stock
WH-04,Bluetooth Speaker,Houston Fulfillment,Satisfied
WH-04,Fountain Pen,Houston Fulfillment,out of stock
WH-05,Notebook Journal,Phoenix Storage,Satisfied


In [0]:
final_df.write.format("delta").mode("overwrite").save("/Volumes/course_enrollment/inventory_schema/filestorage/inventory_details")